In [2]:
import os
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    os.chdir('..')
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Taiwei\RestaurantBookings


In [3]:
import pandas as pd
import duckdb

In [6]:
orders = pd.read_csv('./data/interim/train_silver.csv')
restaurants = pd.read_csv('./data/interim/restaurant_silver.csv')
members = pd.read_csv('./data/interim/member_silver.csv')

In [13]:
orders.columns

Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')

如果忘記git lfs pull會像下面這樣
```
Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')
```

In [9]:
joined_sql = """
    SELECT
        o.booking_id,
        o.member_id,
        o.cdate,
        o.restaurant_id,
        o.datetime,
        o.people,
        o.purpose,
        o.gender,
        o.status,
        o.is_required_prepay_satisfied,
        o.return90,
        r.is_hotel,
        r.city,
        r.cityarea,
        r.good_for_family,
        r.accept_credit_card,
        r.parking,
        r.outdoor_seating,
        r.wifi,
        r.wheelchair_accessible,
        r.price1,
        r.price2,
        r.lat,
        r.lng,
        m.is_vip,
        m.gender AS member_gender,
        m.birthdate,
        m.city AS member_city
    FROM orders o
    LEFT JOIN restaurants r ON o.restaurant_id = r.id
    LEFT JOIN members m ON o.member_id = m.id
"""

In [10]:
joined_df = duckdb.query(joined_sql).to_df()

BinderException: Binder Error: Table "o" does not have a column named "restaurant_id"

Candidate bindings: : "version https://git-lfs.github.com/spec/v1"

In [26]:
joined_df.to_csv('./data/interim/joined_data.csv', index=False)

In [23]:
joined_df.head()

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,...,wifi,wheelchair_accessible,price1,price2,lat,lng,is_vip,member_gender,birthdate,city_1
0,33323,33172341272b9063db8a8a9bc1447604e7b9891f,"7/6/2014,,4:11:00,PM",618b6e3fd20d3f9ba201dede950eeda35f9cb101,"7/7/2014,,6:30:00,PM",6,生日慶祝,M,canceled,1,...,1,0,280,650,24.23,120.94,0,M,0000-00-00,0
1,145160,ddfb8cc2ecaeacbd895c91b1bb29b0f29d9e2e3a,"7/6/2014,,6:01:00,PM",f3837c4489cc6c4587dfb8a92a060585c182ccc1,"7/21/2014,,7:00:00,PM",8,生日慶祝,F,ok,1,...,1,1,713,976,25.05,121.52,0,F,7/21/1983,0
2,38021,3a2cd8c50c1f05604b3c4b873882c1445a680b92,"7/7/2014,,12:53:00,AM",8d448aaa6b7f108f437fa0108454027272da4368,"7/27/2014,,2:20:00,PM",4,生日慶祝,F,canceled,1,...,1,1,589,778,22.67,120.31,0,F,12/15/1988,屏東縣
3,98560,96fb6ed229cec672ea9eaedc5a9025307d6873c1,"9/1/2014,,1:42:00,AM",085996c4fee1b5195077fedd3a2243f95d5d8937,"9/19/2014,,6:00:00,PM",2,None,F,ok,1,...,1,1,1132,2454,25.03,121.56,0,F,8/8/1986,台北市
4,157487,f0f2d1a10a4e85efe1c8669dc5f4f73ddcc47d89,"7/8/2014,,3:38:00,AM",f7ad4a8663cc07948e6047d3477b88c5ce4441ed,"7/18/2014,,6:00:00,PM",2,None,M,ok,1,...,1,1,1500,2000,25.04,121.57,0,M,0000-00-00,0


In [24]:
import ydata_profiling as yp
profile = yp.ProfileReport(joined_df, title="Joined Data Profiling Report", explorative=True)
profile.to_file("./reports/joined_data_profile.html")

d:\Taiwei\RestaurantBookings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 17.39it/s]
